In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
import os
import numpy as np
from sklearn.utils.class_weight import compute_class_weight


In [2]:
# ------------------ GPU CHECK ------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU is available:", gpus)
else:
    print("⚠️ GPU not detected. Training will be slow.")

✅ GPU is available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ------------------ PATHS ------------------
train_dir = "/Users/koushal/Desktop/desktop/ml2 project/dataset/train"
val_dir = "/Users/koushal/Desktop/desktop/ml2 project/dataset/val"

In [4]:
import cv2
import numpy as np

def preprocess_face(img, target_size=(48,48)):
    """
    Preprocessing pipeline for Emotion Detection (Combo 1: CLAHE Boost)
    
    Steps:
    1. Face Detection + Cropping
    2. Grayscale Conversion
    3. CLAHE (Contrast Limited Adaptive Histogram Equalization)
    4. Resize
    5. Normalize (0–1)
    """
    
    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Load Haarcascade face detector
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    
    # Detect faces
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    
    # If no face detected, return None
    if len(faces) == 0:
        return None
    
    # Take the first detected face
    (x, y, w, h) = faces[0]
    face = gray[y:y+h, x:x+w]
    
    # Apply CLAHE (boosts contrast in varying lighting)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    face_clahe = clahe.apply(face)
    
    # Resize to target size
    face_resized = cv2.resize(face_clahe, target_size)
    
    # Normalize to [0,1]
    face_normalized = face_resized.astype("float32") / 255.0
    
    # Expand dims → (48,48,1) for CNN input
    face_final = np.expand_dims(face_normalized, axis=-1)
    
    return face_final

In [5]:
# ------------------ DATA AUGMENTATION ------------------
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    shear_range=0.05,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

Found 8180 images belonging to 3 classes.
Found 1760 images belonging to 3 classes.


In [6]:
 #------------------ CLASS WEIGHTS (to handle imbalance) ------------------
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))
print("📊 Class Weights:", class_weights)


📊 Class Weights: {0: 1.0446998722860792, 1: 0.9682765151515151, 2: 0.9900750423626241}


In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

# ------------------ CNN MODEL ------------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    GlobalAveragePooling2D, Dense, Dropout
)
from tensorflow.keras import regularizers

def build_strong_cnn(input_shape=(128,128,3), num_classes=3):
    weight_decay = 1e-4

    model = Sequential()

    # Block 1
    model.add(Conv2D(32, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay),
                     input_shape=input_shape))
    model.add(BatchNormalization())
    model.add(Conv2D(32, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(0.25))

    # Block 2
    model.add(Conv2D(64, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(Conv2D(64, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(0.3))

    # Block 3
    model.add(Conv2D(128, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(Conv2D(128, (3,3), activation='relu', padding='same',
                     kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(MaxPooling2D((2,2)))
    model.add(Dropout(0.4))

    # Global Average Pooling instead of Flatten
    model.add(GlobalAveragePooling2D())

    # Dense layers
    model.add(Dense(256, activation='relu',
                    kernel_regularizer=regularizers.l2(weight_decay)))
    model.add(BatchNormalization())
    model.add(Dropout(0.5))

    model.add(Dense(num_classes, activation='softmax'))

    # Compile
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    
    return model

# Build model
model = build_strong_cnn(input_shape=(128,128,3), num_classes=3)
model.summary()
# ------------------ BUILD MODEL ------------------
model = build_strong_cnn(
    input_shape=(128, 128, 3),
    num_classes=train_generator.num_classes
)
# ------------------ CALLBACKS ------------------
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath='best_model.keras',   # ✅ use new format
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]
# ------------------ TRAINING ------------------
EPOCHS = 25
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_41 (Conv2D)              │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_46          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_42 (Conv2D)              │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_47          │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_20 (MaxPooling2D) │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_43 (Conv2D)              │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_48          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_44 (Conv2D)              │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_49          │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_45 (Conv2D)              │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_50          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_46 (Conv2D)              │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_51          │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_28 (Dropout)            │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_52          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │             

 Total params: 323,619 (1.23 MB)

 Trainable params: 322,211 (1.23 MB)

 Non-trainable params: 1,408 (5.50 KB)

Epoch 1/25


2025-08-29 18:47:19.698765: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:961] PluggableGraphOptimizer failed: INVALID_ARGUMENT: Failed to deserialize the `graph_buf`.


256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.3651 - loss: 1.6481
Epoch 1: val_accuracy improved from -inf to 0.33466, saving model to best_model.keras
256/256 ━━━━━━━━━━━━━━━━━━━━ 48s 175ms/step - accuracy: 0.3651 - loss: 1.6477 - val_accuracy: 0.3347 - val_loss: 1.7214 - learning_rate: 0.0010
Epoch 2/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 0.3787 - loss: 1.3473
Epoch 2: val_accuracy improved from 0.33466 to 0.39091, saving model to best_model.keras
256/256 ━━━━━━━━━━━━━━━━━━━━ 45s 175ms/step - accuracy: 0.3787 - loss: 1.3471 - val_accuracy: 0.3909 - val_loss: 1.1397 - learning_rate: 0.0010
Epoch 3/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - accuracy: 0.3950 - loss: 1.2176
Epoch 3: val_accuracy improved from 0.39091 to 0.42841, saving model to best_model.keras
256/256 ━━━━━━━━━━━━━━━━━━━━ 45s 177ms/step - accuracy: 0.3950 - loss: 1.2175 - val_accuracy: 0.4284 - val_loss: 1.1211 - learning_rate: 0.0010
Epoch 4/25
256/256 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/st